# 01b - Does Beating the Spread Carry Over?

**Question:** If a team beat the point spread in its last game, is it more or less
likely to beat the spread in its next game?

**Why it matters:** This is a small test of market efficiency. The point spread
should already account for everything known before kickoff, including last week's
result. If past performance against the spread predicts future performance, the
market is systematically over- or under-reacting, and that would be a candidate
feature for the Phase 2 margin model. If it doesn't, the feature hasn't earned
its place.

**Requires:** `data/games_clean.parquet`. Run `01_data_exploration.ipynb` first.

## Hypotheses (stated before looking at results)

1. **Efficient market (null):** last game's result against the spread has no
   relationship with this game's. Cover rates stay near 50% either way.
2. **Overreaction:** teams that just beat the spread get overrated, so their next
   line is too generous and they cover *less* often than 50%.
3. **Momentum:** teams that just beat the spread are improving faster than the
   market adjusts, so they cover *more* often than 50%.

**Main test (decided in advance):** cover rate in a team's next game, split by
whether it covered its previous game. All other cuts are exploratory.

**Benchmarks for judging results:**
- Statistical: with ~3,500 team-games per group, differences under ~2 percentage
  points are within noise.
- Financial: at standard −110 odds, a strategy must win 52.4% of bets to break even.

## Contents
1. Setup and load
2. Build the team-game table (one row per team per game)
3. Attach each team's previous game
4. Main test: cover rate by previous cover
5. Exploratory: correlation and blowout buckets
6. Robustness: home-team rows only; 2012–2021 vs. 2022–2025
7. Findings and decisions

## 1. Setup and load

In [3]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 

clean = pd.read_parquet("../data/games_clean.parquet")
clean.shape

(3829, 21)

In [5]:
home = pd.DataFrame({
    "game_id": clean["game_id"],
    "season": clean["season"],
    "week": clean["week"],
    "kickoff": clean["kickoff"],
    "is_playoff": clean["is_playoff"], 
    "team": clean["home_team"],
    "opponent": clean["away_team"],
    "is_home": True,
    "margin": clean["result"],                  # team's points minus oppoent's
    "expected_margin": clean["spread_line"]     # market's expected margin for this team
})

In [6]:
away = pd.DataFrame({
    "game_id": clean["game_id"],
    "season": clean["season"],
    "week": clean["week"],
    "kickoff": clean["kickoff"],
    "is_playoff": clean["is_playoff"], 
    "team": clean["away_team"],
    "opponent": clean["home_team"],
    "is_home": False,
    "margin": -clean["result"],
    "expected_margin": -clean["spread_line"]
})

In [8]:
tg = pd.concat([home, away], ignore_index = True)
tg.sort_values(["team", "kickoff"]).reset_index(drop=True)

# How much the team beat the spread by, from its own side
tg["ats_margin"] = tg["margin"] - tg["expected_margin"]

# 1 = covered, 0 = didn't, NaN = push (landed exactly on the spread)
tg["covered"] = (tg["ats_margin"] > 0).astype(float).where(tg["ats_margin"] != 0)

tg.head(20)

,game_id,season,week,kickoff,is_playoff,team,opponent,is_home,margin,expected_margin,ats_margin,covered
0,2012_01_DAL_NYG,2012,1,2012-09-05 20:30:00,False,NYG,DAL,True,-7,3.5,-10.5,0.0
1,2012_01_IND_CHI,2012,1,2012-09-09 13:00:00,False,CHI,IND,True,20,10.0,10.0,1.0
2,2012_01_PHI_CLE,2012,1,2012-09-09 13:00:00,False,CLE,PHI,True,-1,-9.0,8.0,1.0
3,2012_01_STL_DET,2012,1,2012-09-09 13:00:00,False,DET,LA,True,4,9.0,-5.0,0.0
4,2012_01_MIA_HOU,2012,1,2012-09-09 13:00:00,False,HOU,MIA,True,20,13.0,7.0,1.0
5,2012_01_ATL_KC,2012,1,2012-09-09 13:00:00,False,KC,ATL,True,-16,-1.0,-15.0,0.0
6,2012_01_JAX_MIN,2012,1,2012-09-09 13:00:00,False,MIN,JAX,True,3,3.5,-0.5,0.0
7,2012_01_WAS_NO,2012,1,2012-09-09 13:00:00,False,NO,WAS,True,-8,9.5,-17.5,0.0
8,2012_01_BUF_NYJ,2012,1,2012-09-09 13:00:00,False,NYJ,BUF,True,20,3.0,17.0,1.0
9,2012_01_NE_TEN,2012,1,2012-09-09 13:00:00,False,TEN,NE,True,-21,-4.5,-16.5,0.0


In [9]:
# 1. Every game appears exactly twice
assert len(tg) == 2 * len(clean), "Row count should double"
assert (tg["game_id"].value_counts() == 2).all(), "Some game isn't exactly twice"

# 2. The two sides mirror each other: ATS margins in a game sum to zero
ats_sum = tg.groupby("game_id")["ats_margin"].sum()
assert (ats_sum.abs() < 1e-9).all(), "Home and away ATS margins don't mirror"

# 3. In every non-push game, exactly one team covered
cov_sum = tg.dropna(subset=["covered"]).groupby("game_id")["covered"].sum()
assert (cov_sum == 1).all(), "Both or neither team covered in some game"

print("All checks passed")
print(f"Team-games: {len(tg)}, pushes: {tg['covered'].isna().sum() // 2} games")

All checks passed
Team-games: 7658, pushes: 93 games


In [13]:
tg[(tg["team"] == "CAR") & (tg["season"] == 2023)][
    ["week", "opponent", "is_home", "margin", "expected_margin", "ats_margin", "covered"]
].sort_values("week")

,week,opponent,is_home,margin,expected_margin,ats_margin,covered
6810,1,ATL,False,-14,-3.5,-10.5,0.0
3004,2,NO,True,-3,-3.0,0.0,NaN
6845,3,SEA,False,-10,-5.0,-5.0,0.0
3028,4,MIN,True,-8,-4.5,-3.5,0.0
6873,5,DET,False,-18,-9.5,-8.5,0.0
6884,6,MIA,False,-21,-14.0,-7.0,0.0
3088,8,HOU,True,2,-3.0,5.0,1.0
3105,9,IND,True,-14,-1.5,-12.5,0.0
6939,10,CHI,False,-3,-3.0,0.0,NaN
3131,11,DAL,True,-23,-11.0,-12.0,0.0


From the table above, we can see that the Panthers were terrible in 2023